## AutoCast end-to-end training and evaluation example

This notebook demonstrates end-to-end training of:

- autoencoder
- flow matching
- evaluation


### Example dataset

We use autosim to generate a small 32x32 advection-diffusion dataset with train/valid/test splits, cached as `data.pt` files plus `stats.yml`.


In [ ]:
from pathlib import Path

from autocast.metrics import MAE, MSE, RMSE

THE_WELL = False
simulation_name = "advection_diffusion_multichannel"
spatial_resolution = 32
n_steps_input = 1
n_steps_output = 4
stride = n_steps_output
rollout_stride = n_steps_output

n_train = 50
n_valid = 4
n_test = 4
batch_size = 4

max_epochs_autoencoder = 5
max_epochs_processor = 10
accelerator = "auto"

the_well_dataset_path = "../datasets/"
overwrite_data = False
data_path = (
    Path("../datasets/tmp")
    / f"{simulation_name}_{spatial_resolution}x{spatial_resolution}"
)
simulator_kwargs = {"n": spatial_resolution, "T": 25.0, "dt": 0.25}

### Read data into datamodules


In [ ]:
from autocast.data.utils import get_autosim_datamodule, get_datamodule

datamodule_kwargs = {
    "simulation_name": simulation_name,
    "n_steps_input": n_steps_input,
    "n_steps_output": n_steps_output,
    "stride": stride,
    "n_train": n_train,
    "n_valid": n_valid,
    "n_test": n_test,
    "data_path": str(data_path),
    "simulator_kwargs": simulator_kwargs,
    "batch_size": batch_size,
    "num_workers": 0,
    "use_normalization": True,
}

if THE_WELL:
    well_datamodule_kwargs = {
        "the_well": True,
        "simulation_name": simulation_name,
        "n_steps_input": n_steps_input,
        "n_steps_output": n_steps_output,
        "stride": stride,
        "the_well_dataset_path": the_well_dataset_path,
        "batch_size": batch_size,
        "num_workers": 0,
        "use_normalization": True,
    }
    ae_datamodule = get_datamodule(
        **well_datamodule_kwargs,
        autoencoder_mode=True,
    )
    datamodule = get_datamodule(
        **well_datamodule_kwargs,
        autoencoder_mode=False,
    )
else:
    ae_datamodule = get_autosim_datamodule(
        **datamodule_kwargs,
        autoencoder_mode=True,
        overwrite=overwrite_data,
    )
    datamodule = get_autosim_datamodule(
        **datamodule_kwargs,
        autoencoder_mode=False,
    )


### Set-up logging


In [ ]:
from autocast.logging import maybe_watch_model
from autocast.logging.wandb import create_notebook_logger

logger, watch = create_notebook_logger(
    project="autocast-notebooks",
    name=f"04_e2e_{simulation_name}",
    tags=["notebook", simulation_name],
    enabled=False,
)
trainer_logger = logger if logger is not None else False

In [ ]:
batch = next(iter(datamodule.train_dataloader()))
n_channels = batch.input_fields.shape[-1]
w, h = batch.input_fields.shape[2:4]

In [ ]:
# Compress by a factor of 4, with the spatial autoencoder reducing 32x32
# fields to 16x16 latents by default.
compression = 4
latent_width, latent_height = w // 2, h // 2
n_latent = max(
    1,
    (w * h * n_channels // (latent_width * latent_height) // compression),
)

print(f"n_latent channels equals {n_latent} for a compression factor of {compression}")


In [ ]:
# Train autoencoder
from autocast.decoders.dc import DCDecoder
from autocast.encoders.dc import DCEncoder
from autocast.models.autoencoder import AE
from autocast.utils import get_optimizer_config

encoder = DCEncoder(
    in_channels=n_channels,
    out_channels=n_latent,
    hid_channels=[32, 64],
    hid_blocks=[1, 1],
    kernel_size=3,
    stride=2,
    spatial=2,
    pixel_shuffle=False,
    periodic=False,
    dropout=None,
)
decoder = DCDecoder(
    in_channels=n_latent,
    out_channels=n_channels,
    hid_channels=[64, 32],
    hid_blocks=[1, 1],
    kernel_size=3,
    stride=2,
    spatial=2,
    pixel_shuffle=False,
    periodic=False,
    dropout=None,
)

ae = AE(encoder=encoder, decoder=decoder, optimizer_config=get_optimizer_config(5e-4))

In [ ]:
encoded, global_cond = ae.encoder.encode_with_cond(next(iter(ae_datamodule.train_dataloader())))


In [ ]:
print("Encoded shape is:", tuple(encoded.shape))

In [ ]:
import lightning as L

trainer = L.Trainer(
    max_epochs=max_epochs_autoencoder,
    accelerator=accelerator,
    devices=1,
    logger=trainer_logger,
    enable_checkpointing=False,
)
trainer.fit(ae, ae_datamodule)
trainer.save_checkpoint(f"./{simulation_name}_ae_model.ckpt")


In [ ]:
import matplotlib.pyplot as plt

device = "cpu"
ae = ae.to(device)
num_examples = 2
for idx, batch in enumerate(ae_datamodule.test_dataloader()):
    batch = batch.to(device)
    inputs = batch.input_fields
    outputs, latents = ae.forward_with_latent(batch)
    print("Input shape:", inputs.shape)
    print("Output shape:", outputs.shape)
    print("Latent shape:", latents.shape)
    input_frame = inputs[0, 0, :, :, 0].detach().cpu().numpy()
    output_frame = outputs[0, 0, :, :, 0].detach().cpu().numpy()
    latent_frame = latents[0, 0, :, :, 0].detach().cpu().numpy()

    fig, axs = plt.subplots(1, 3, figsize=(12, 4))
    axs[0].imshow(input_frame, cmap="viridis")
    axs[0].set_title("Input")
    axs[1].imshow(output_frame, cmap="viridis")
    axs[1].set_title("Reconstruction")
    axs[2].imshow(latent_frame, cmap="viridis")
    axs[2].set_title("Latent")
    plt.show()

    if idx + 1 >= num_examples:
        break


### Example shape and batch


In [ ]:
datamodule.train_dataset[0].input_fields.shape

In [ ]:
batch = next(iter(datamodule.train_dataloader()))

batch.input_fields.shape

In [ ]:
from azula.noise import VPSchedule

from autocast.metrics.deterministic import VRMSE
from autocast.models.encoder_processor_decoder import EncoderProcessorDecoder
from autocast.nn.base import TemporalBackboneBase
from autocast.nn.unet import TemporalUNetBackbone
from autocast.nn.vit import TemporalViTBackbone
from autocast.processors.diffusion import DiffusionProcessor
from autocast.processors.flow_matching import FlowMatchingProcessor
from autocast.utils import get_optimizer_config

# Get sample batch
batch = next(iter(datamodule.train_dataloader()))
encoded, example_global_cond = encoder.encode_with_cond(batch)
latent_channels = encoded.shape[-1]
global_cond_channels = (
    example_global_cond.shape[-1]
    if example_global_cond is not None
    else None
)
include_global_cond = global_cond_channels is not None


def get_backbone(backbone_name: str) -> TemporalBackboneBase:
    """Create backbone based on name."""
    if backbone_name == "unet":
        backbone = TemporalUNetBackbone(
            in_channels=latent_channels,
            out_channels=latent_channels,
            cond_channels=latent_channels,
            n_steps_output=n_steps_output,
            n_steps_input=n_steps_input,
            include_global_cond=include_global_cond,
            global_cond_channels=global_cond_channels,
            mod_features=64,
            hid_channels=(16, 32),
            hid_blocks=(1, 1),
            spatial=2,
            periodic=False,
        )
    elif backbone_name == "vit":
        backbone = TemporalViTBackbone(
            in_channels=latent_channels,
            out_channels=latent_channels,
            cond_channels=latent_channels,
            n_steps_output=n_steps_output,
            n_steps_input=n_steps_input,
            mod_features=256,
            include_global_cond=include_global_cond,
            global_cond_channels=global_cond_channels,
            hid_channels=160,
            hid_blocks=10,
            temporal_method="none",
            attention_heads=4,
            spatial=2,
            patch_size=4,
            dropout=0.05,
            ffn_factor=4,
            checkpointing=False,
        )
    else:
        raise ValueError(f"Unknown backbone name: {backbone_name}")
    return backbone


backbone_name = "vit"  # set to "unet" or "vit"
backbone = get_backbone(backbone_name)


# Construct processor
processor_name = "flow_matching"  # set to "diffusion" to compare
# processor_name = "diffusion"  # set to "flow_matching" to compare

if processor_name == "flow_matching":
    processor = FlowMatchingProcessor(
        backbone=backbone,
        n_steps_output=n_steps_output,
        n_channels_out=latent_channels,
        flow_ode_steps=4,
    )
else:
    processor = DiffusionProcessor(
        backbone=backbone,
        schedule=VPSchedule(),
        n_steps_output=n_steps_output,
        n_channels_out=latent_channels,
    )

model = EncoderProcessorDecoder(
    encoder_decoder=ae,
    processor=processor,
    train_in_latent_space=True,
    optimizer_config=get_optimizer_config(5e-4),
    val_metrics=[VRMSE(), MSE(), MAE(), RMSE()],
    test_metrics=[VRMSE(), MSE(), MAE(), RMSE()],
)
maybe_watch_model(logger, model, watch)


### Run trainer


In [ ]:
import lightning as L

trainer = L.Trainer(
    max_epochs=max_epochs_processor,
    accelerator=accelerator,
    devices=1,
    logger=trainer_logger,
    enable_checkpointing=False,
)
trainer.fit(model, datamodule)
trainer.save_checkpoint(f"./{simulation_name}_{processor_name}_model.ckpt")


### Run the evaluation


In [ ]:
trainer.test(model, datamodule)

### Example rollout


In [ ]:
# A single element is the full trajectory
batch = next(iter(datamodule.rollout_test_dataloader(batch_size=1)))


In [ ]:
# First n_steps_input are inputs
print(batch.input_fields.shape)
# Remaining n_steps_output are outputs
print(batch.output_fields.shape)

In [ ]:
from autocast.models.encoder_processor_decoder_ensemble import (
    EncoderProcessorDecoderEnsemble,
)

ensemble_model = EncoderProcessorDecoderEnsemble(
    encoder_decoder=model.encoder_decoder,
    processor=model.processor,
    train_in_latent_space=False,
    optimizer_config=get_optimizer_config(5e-4),
    test_metrics=[],
    val_metrics=[],
    n_members=5,
    batch_size=2,
)


In [ ]:
# Run rollout on one trajectory
ensemble_model = ensemble_model.to("cpu")
batch = batch.to("cpu")
preds, trues = ensemble_model.rollout(
    batch,
    stride=rollout_stride,
    max_rollout_steps=50,
    free_running_only=True,
    n_members=5,
)

print(preds.shape)  # B, T, H, W, C, M
assert trues is not None
print(trues.shape)  # B, T, H, W, C


In [ ]:
from autocast.metrics import MSE

assert trues is not None
pred_mean = preds.mean(dim=-1)
assert pred_mean.shape == trues.shape
mse = MSE()
mse_error = mse(pred_mean, trues)
print("MSE overall is a single scalar:", mse_error)


In [ ]:
from IPython.display import HTML

from autocast.utils import plot_spatiotemporal_video

batch_idx = 0
channel_names = (
    datamodule.train_dataset.normalization_stats.get("core_field_names")
    if getattr(datamodule.train_dataset, "normalization_stats", None) is not None
    else None
)

anim = plot_spatiotemporal_video(
    pred=pred_mean,
    true=trues,
    pred_uq=preds.std(dim=-1),
    batch_idx=batch_idx,
    save_path=f"{simulation_name}_{batch_idx:02d}.mp4",
    colorbar_mode="column",
    channel_names=channel_names,
    pred_uq_label="Ensemble Std. Dev.",
    colorbar_mode_uq="row",
)


In [ ]:
# Plot in notebook
HTML(anim.to_jshtml())